# 08 · Master Pipeline Kaggle – Zero-Trust / Zero-Persistent-Image

Pipeline de geração com **isolamento total de I/O em memória RAM (tmpfs)**:

| Caminho | Localização | Volátil? |
|---|---|---|
| INPUT (pastes, uploads, img2img) | `/dev/shm/comfy_ui_input` | ✅ RAM |
| OUTPUT (imagens geradas) | `/dev/shm/comfy_ui_output` | ✅ RAM |
| TEMP (latentes, previews) | `/dev/shm/comfy_ui_temp` | ✅ RAM |
| ZIP de download | `/dev/shm/comfy_ui_archive/output.zip` | ✅ RAM |

**Nenhuma imagem toca `/kaggle/working` em nenhum momento controlado pelo pipeline.**

O ZIP de download é criptografado com AES-256 via `pyzipper` e requer o Secret `SECRET_ZIP_PASSWORD`.

**Limites desta arquitetura:** não controla a infraestrutura do Kaggle, acesso privilegiado do provedor
ao host, nem vulnerabilidades em dependências externas. Previne persistência acidental nos caminhos
controlados pelo pipeline com falha explícita (fail-closed).

## Fluxo de cada sessão
1. Validar Dataset montado em `/kaggle/input/<slug>`
2. Sincronizar repositório GitHub
3. Detectar GPU
4. Configurar Google Drive (opcional, backup)
5. Provisionar tmpfs + instalar ComfyUI
6. **Verificação de segurança:** INPUT/OUTPUT/TEMP em `/dev/shm` (fail-closed)
7. Iniciar ComfyUI com processo novo (reuse_existing=False)
8. Gerar imagens (outputs em RAM)
9. Coletar → Criptografar → ZIP em `/dev/shm` → Download → Apagar
10. **Verificação final:** zero imagens em `/kaggle/working`

In [ ]:
from pathlib import Path
import subprocess, sys, os, time, json

REPO_URL = "https://github.com/automadevs/colab-pipeline.git"
WORKDIR = Path("/kaggle/working")
REPO_DIR = WORKDIR / "colab-pipeline"
SCRIPTS_DIR = WORKDIR / "scripts"
COMFYUI_DIR = WORKDIR / "ComfyUI"

# Zero-Persistent-Image: TODO I/O de imagem em tmpfs (RAM volátil)
# /dev/shm NÃO é varrido pelo snapshot/versionamento do Kaggle
SHM_INPUT   = Path("/dev/shm/comfy_ui_input")
SHM_OUTPUT  = Path("/dev/shm/comfy_ui_output")
SHM_TEMP    = Path("/dev/shm/comfy_ui_temp")
SHM_ARCHIVE = Path("/dev/shm/comfy_ui_archive")
SECURE_ZIP  = SHM_ARCHIVE / "output.zip"   # NUNCA em /kaggle/working

# Modelos locais (graváveis, somente binários de modelo — sem imagens)
MODELS_DIR = COMFYUI_DIR / "models"

def _get_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def resolve_dataset_name(override=None):
    """Resolve o dataset Kaggle: override explícito ou KAGGLE_USERNAME/KAGGLE_DATASET_NAME."""
    if override:
        return override
    username = _get_secret("KAGGLE_USERNAME")
    dataset_name = _get_secret("KAGGLE_DATASET_NAME")
    if not username or not dataset_name:
        raise ValueError(
            "Dataset Kaggle não resolvido. Configure os Secrets do Kaggle:\n"
            "  - KAGGLE_USERNAME: seu username Kaggle\n"
            "  - KAGGLE_DATASET_NAME: nome do dataset (ex: comfydocs)\n"
            "Ou defina DATASET_OVERRIDE no início desta célula."
        )
    return f"{username}/{dataset_name}"

DATASET_OVERRIDE = None  # ex: "meuusuario/meudataset" para forçar
DATASET = resolve_dataset_name(DATASET_OVERRIDE)
print(f"[INFO] Dataset alvo: {DATASET}")

DRIVE_BASE = "Automa/ComfyUI"

In [ ]:
# Dataset anexado como INPUT do notebook (Add Data na UI do Kaggle) -- leitura direta,
# sem download/cópia para o SSD. O ComfyUI lê o Dataset montado via extra_model_paths.yaml.
DATASET_SLUG = _get_secret("KAGGLE_DATASET_NAME")
if not DATASET_SLUG:
    raise RuntimeError(
        "Secret KAGGLE_DATASET_NAME não configurado -- necessário para localizar "
        "o mount do dataset em /kaggle/input."
    )

DATASET_INPUT_DIR = Path("/kaggle/input") / DATASET_SLUG
if not DATASET_INPUT_DIR.is_dir():
    raise RuntimeError(
        f"Dataset '{DATASET}' não está anexado como Input deste notebook "
        f"(esperado em {DATASET_INPUT_DIR}). No painel direito do Kaggle: "
        f"Add Input -> Datasets -> busque '{DATASET}' -> Add."
    )

print(f"[INFO] Dataset montado (somente leitura) em: {DATASET_INPUT_DIR}")

In [ ]:
# Sincroniza sempre o repositório como fonte da verdade e copia scripts para execução
import importlib
import shutil

if REPO_DIR.exists():
    print("[INFO] Atualizando repositório via git pull...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    print("[INFO] Clonando repositório...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

# Copiar scripts atualizados do checkout garantindo que versões órfãs sejam eliminadas
if SCRIPTS_DIR.exists():
    shutil.rmtree(SCRIPTS_DIR)
shutil.copytree(REPO_DIR / "scripts", SCRIPTS_DIR)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

importlib.invalidate_caches()
for module_name in ("comfyui_setup", "ngrok_tunnel", "gpu_detect", "kaggle_drive_sync"):
    module = sys.modules.get(module_name)
    if module is not None:
        importlib.reload(module)

print("[INFO] Scripts atualizados disponíveis:", sorted(p.name for p in SCRIPTS_DIR.glob("*.py")))

In [ ]:
# Detecção e validação de GPU NVIDIA
from gpu_detect import detect_gpu
GPU_INFO = detect_gpu()
print(json.dumps(GPU_INFO, indent=2, ensure_ascii=False))
if not GPU_INFO.get("has_gpu"):
    raise RuntimeError("GPU NVIDIA não detectada. Ative Accelerator → GPU no Kaggle antes de continuar.")

print("=== GPU / DISCO ===")
subprocess.run(["nvidia-smi"], check=False)
subprocess.run(["df", "-h", str(WORKDIR)], check=False)
subprocess.run(["df", "-h", "/dev/shm"], check=False)

In [ ]:
# Configuração do Google Drive SOMENTE para persistência/backup.
# Credenciais são apagadas após montagem.
from kaggle_drive_sync import test_drive_connection, cleanup_rclone_credentials

print("=" * 60)
print("CONFIGURANDO GOOGLE DRIVE (PERSISTÊNCIA/BACKUP)")
print("=" * 60)

DRIVE_AVAILABLE = False
try:
    test_res = test_drive_connection(drive_base=DRIVE_BASE, env="kaggle")
    if test_res["status"] == "pass":
        DRIVE_AVAILABLE = True
        DRIVE_PATH = Path(test_res["drive_path"])
        print(f"[INFO] Google Drive pronto para sync em: {DRIVE_PATH}")
        # Apagar credenciais após montagem bem-sucedida
        cleanup_rclone_credentials()
    else:
        print(f"[WARN] Google Drive não pôde ser montado: {test_res.get('error')}")
except Exception as e:
    print(f"[WARN] Falha ao configurar Google Drive: {e}")
    print("[INFO] O ComfyUI funcionará normalmente gerando no SSD local.")


In [ ]:
# Provisiona tmpfs e instala/atualiza ComfyUI + custom nodes
# INPUT/OUTPUT/TEMP → /dev/shm (tmpfs volátil)
from comfyui_setup import setup_comfyui, provision_shm_dirs, check_custom_nodes_allowlist

CUSTOM_NODES = [
    "cubiq/ComfyUI_essentials",
    "lbouaraba/comfyui-krea2edit",
]

# Provisionar TODOS os diretórios tmpfs com permissão total
provision_shm_dirs(SHM_INPUT, SHM_OUTPUT, SHM_TEMP, SHM_ARCHIVE)
subprocess.run(["df", "-h", "/dev/shm"], check=False)

setup_comfyui(
    comfyui_dir=COMFYUI_DIR,
    models_dir=MODELS_DIR,
    custom_nodes=CUSTOM_NODES,
    output_dir=SHM_OUTPUT,
    input_dir=SHM_INPUT,
    temp_dir=SHM_TEMP,
    additional_model_roots=[("dataset_models", DATASET_INPUT_DIR)],
    strict_allowlist=True,
)
print(f"[SECURITY] INPUT  → {SHM_INPUT} (tmpfs)")
print(f"[SECURITY] OUTPUT → {SHM_OUTPUT} (tmpfs)")
print(f"[SECURITY] TEMP   → {SHM_TEMP} (tmpfs)")
print(f"[SECURITY] ARCHIVE→ {SHM_ARCHIVE} (tmpfs)")
print(f"[INFO] Modelos locais em: {MODELS_DIR}")
print(f"[INFO] Modelos Dataset em: {DATASET_INPUT_DIR} (somente leitura)")

In [ ]:
# =============================================================
# VERIFICAÇÃO DE SEGURANÇA — fail-closed
# Valida que INPUT/OUTPUT/TEMP apontam para /dev/shm
# =============================================================
from comfyui_setup import assert_shm_path, SecurityError, final_filesystem_check

print("=== COMFYUI SECURITY CONFIGURATION ===")
print(f"INPUT   = {SHM_INPUT}")
print(f"TEMP    = {SHM_TEMP}")
print(f"OUTPUT  = {SHM_OUTPUT}")
print(f"ARCHIVE = {SHM_ARCHIVE}")
print("=== END SECURITY CONFIGURATION ===")

errors = []
for label, path in [("INPUT", SHM_INPUT), ("OUTPUT", SHM_OUTPUT),
                    ("TEMP", SHM_TEMP), ("ARCHIVE", SHM_ARCHIVE)]:
    if not str(path).startswith("/dev/shm"):
        errors.append(f"{label} está fora de /dev/shm: {path}")
    if "/kaggle/working" in str(path):
        errors.append(f"{label} aponta para /kaggle/working: {path}")

if errors:
    raise SecurityError(
        "SECURITY VIOLATION: Paths não estão em /dev/shm:\n"
        + "\n".join(f"  - {e}" for e in errors)
    )

# Verificação inicial: /kaggle/working não deve ter imagens antes de começar
pre_check = final_filesystem_check(silent=False)
if pre_check["images"]:
    raise SecurityError(
        f"SECURITY VIOLATION: {len(pre_check['images'])} imagem(ns) encontrada(s) "
        "em /kaggle/working antes de iniciar. Limpe o ambiente e reinicie o kernel."
    )

print("\n[SECURITY] ✅ Todos os paths validados — pipeline pode continuar")

In [ ]:
# Inicia ComfyUI com processo NOVO (reuse_existing=False)
# ngrok desabilitado por padrão — habilite explicitamente abaixo se necessário
from comfyui_setup import start_comfyui_runtime

COMFYUI_PORT = 8188
ENABLE_NGROK = False  # Altere para True SOMENTE se precisar de acesso externo
                      # (requer Secret NGROK_AUTHTOKEN)

# Reassert defensivo imediatamente antes do start
provision_shm_dirs(SHM_INPUT, SHM_OUTPUT, SHM_TEMP, SHM_ARCHIVE)

runtime = start_comfyui_runtime(
    comfyui_dir=COMFYUI_DIR,
    host="127.0.0.1",
    port=COMFYUI_PORT,
    output_dir=SHM_OUTPUT,
    input_dir=SHM_INPUT,
    temp_dir=SHM_TEMP,
    enable_ngrok=ENABLE_NGROK,
    reuse_existing=False,   # SEMPRE inicia processo novo
)

if not runtime["health"]:
    raise RuntimeError(f"ComfyUI não ficou saudável. Log: {runtime['log_path']}")

print("=" * 60)
print("COMFYUI READY — ZERO-PERSISTENT-IMAGE")
print(f"Local  : {runtime['local_url']}")
print(f"Public : {runtime['public_url'] or '(ngrok desabilitado)'}")
print(f"INPUT  : {SHM_INPUT} (tmpfs)")
print(f"OUTPUT : {SHM_OUTPUT} (tmpfs)")
print(f"TEMP   : {SHM_TEMP} (tmpfs)")
print(f"GPU    : {GPU_INFO.get('gpu_count', 0)} GPU(s)")
print(f"PID    : {runtime.get('pid')}")
print("=" * 60)

In [ ]:
# =============================================================
# GERENCIAMENTO DE CICLO DE VIDA DE ARTEFATOS
# Fluxo: GENERATE → COLLECT → VALIDATE → ENCRYPT → DOWNLOAD → DELETE → VERIFY
# =============================================================
import gc
import urllib.request
from comfyui_setup import create_secure_zip, cleanup_zip, clear_input, clear_output, final_filesystem_check, SecurityError

COMFYUI_API = f"http://127.0.0.1:{COMFYUI_PORT}"

def _api_get(path: str) -> dict:
    with urllib.request.urlopen(f"{COMFYUI_API}{path}", timeout=10) as resp:
        return json.loads(resp.read().decode("utf-8"))

def wait_queue_empty(poll_s: float = 3.0, timeout_s: float = 7200.0) -> None:
    """Bloqueia até a fila do ComfyUI drenar completamente (running=0, pending=0)."""
    start = time.time()
    while True:
        queue = _api_get("/queue")
        running = len(queue.get("queue_running", []))
        pending = len(queue.get("queue_pending", []))
        elapsed = int(time.time() - start)
        print(f"[POLL] running={running} pending={pending} elapsed={elapsed}s   ", end="\r")
        if running == 0 and pending == 0:
            print(f"\n[INFO] Fila drenada após {elapsed}s. Geração concluída.")
            return
        if elapsed > timeout_s:
            raise TimeoutError(f"Fila não drenou em {timeout_s}s — verifique o log do ComfyUI.")
        time.sleep(poll_s)

def report_history_failures() -> None:
    """Audita /history em busca de jobs com erro."""
    history = _api_get("/history")
    failures = 0
    for prompt_id, entry in history.items():
        status = entry.get("status", {})
        if status.get("status_str") == "error" or status.get("completed") is False:
            failures += 1
            print(f"[WARN] Job com falha: prompt_id={prompt_id}")
    if failures == 0:
        print(f"[INFO] Histórico íntegro: {len(history)} job(s), nenhuma falha.")

# Fluxo estrito: COLLECT → VALIDATE → ENCRYPT → (DOWNLOAD MANUAL) → DELETE → VERIFY
wait_queue_empty()
report_history_failures()

# 1. Criptografar e empacotar em /dev/shm
secure_zip_path = create_secure_zip(
    src_dir=SHM_OUTPUT,
    archive_dir=SHM_ARCHIVE,
    zip_name="output.zip",
    # Senha vem do Secret SECRET_ZIP_PASSWORD automaticamente
)
print(f"[SECURITY] ZIP criado em tmpfs: {secure_zip_path}")
print(f"[SECURITY] Faça o download de {secure_zip_path} pela célula abaixo.")

# 2. Limpar output e input após criar o ZIP
clear_output(SHM_OUTPUT)
clear_input(SHM_INPUT)
gc.collect()
print("[CLEANUP] Output e Input limpos da RAM.")

In [ ]:
# =============================================================
# DOWNLOAD DO ZIP — execute esta célula para obter o arquivo
# Após download: execute a célula de cleanup abaixo
# =============================================================
# O ZIP está em /dev/shm/comfy_ui_archive/output.zip
# Para baixar no Kaggle: use o painel Files do notebook e navegue até /dev/shm/comfy_ui_archive/
# OU copie para /kaggle/working TEMPORARIAMENTE para download, depois apague:

import shutil

DOWNLOAD_STAGING = WORKDIR / "output_download.zip"  # Nome diferente de output_secure.zip

if secure_zip_path.exists():
    shutil.copy2(secure_zip_path, DOWNLOAD_STAGING)
    size_mb = DOWNLOAD_STAGING.stat().st_size / (1024**2)
    print(f"[INFO] ZIP copiado para download: {DOWNLOAD_STAGING} ({size_mb:.1f} MB)")
    print(f"[INFO] Após download, execute a célula de CLEANUP abaixo para apagar {DOWNLOAD_STAGING}")
else:
    print("[WARN] ZIP não encontrado — execute a célula de ciclo de vida primeiro")

In [ ]:
# =============================================================
# CLEANUP PÓS-DOWNLOAD — execute APÓS fazer o download do ZIP
# Apaga staging, ZIP tmpfs e verifica filesystem
# =============================================================
from comfyui_setup import safe_remove, cleanup_zip, final_filesystem_check, SecurityError

# Apagar staging em /kaggle/working
if DOWNLOAD_STAGING.exists():
    safe_remove(DOWNLOAD_STAGING)
    print(f"[CLEANUP] ✓ Staging removido: {DOWNLOAD_STAGING}")

# Apagar ZIP de tmpfs
cleanup_zip(secure_zip_path)

# Verificação final
result = final_filesystem_check(scan_root=WORKDIR, silent=False)
if result["violations"] > 0:
    raise SecurityError(
        f"SECURITY VIOLATION: {result['violations']} artefato(s) sensível(is) "
        "encontrado(s) em /kaggle/working após cleanup.\n"
        "Verifique o relatório acima."
    )

print("\n[SECURITY] ✅ Cleanup pós-download concluído — zero artefatos persistentes")

In [ ]:
# =============================================================
# VALIDAÇÃO DE WORKFLOW (CLIPLoader krea2)
# =============================================================
WORKFLOW_JSON = WORKDIR / "lustify_simple_t2i.json"

def validate_workflow_node(path, node_id="9", expected_type="krea2"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Workflow não encontrado: {path}")
    wf = json.loads(path.read_text(encoding="utf-8"))

    node = None
    if node_id in wf and isinstance(wf[node_id], dict):
        node = wf[node_id]
    elif isinstance(wf.get("nodes"), list):
        node = next((n for n in wf["nodes"] if str(n.get("id")) == node_id), None)

    if node is None:
        raise KeyError(f"Nó {node_id} ausente em {path.name}")

    actual = node.get("class_type") or node.get("type")
    if actual != expected_type:
        raise ValueError(
            f"Nó {node_id} com type={actual!r}; esperado {expected_type!r}. "
            "Re-exporte o workflow com o CLIPLoader krea2."
        )
    print(f"[OK] Nó {node_id} ({path.name}): type={actual!r} — workflow compatível")
    return True

validate_workflow_node(WORKFLOW_JSON)

In [ ]:
%%bash
# Validação operacional do isolamento tmpfs
echo "=== USO DO TMPFS (/dev/shm) ==="
df -h /dev/shm
echo
echo "=== FOOTPRINT DOS DIRETÓRIOS VOLÁTEIS ==="
du -sh /dev/shm/comfy_ui_input /dev/shm/comfy_ui_output /dev/shm/comfy_ui_temp /dev/shm/comfy_ui_archive 2>/dev/null || echo "(diretórios ainda não criados)"
echo
echo "=== AUDITORIA ZERO-DISK: nenhuma imagem em /kaggle/working ==="
find /kaggle/working -type f \( -iname "*.png" -o -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.webp" -o -iname "*.gif" -o -iname "*.bmp" -o -iname "*.tif" -o -iname "*.tiff" -o -iname "*.latent" \) 2>/dev/null | head -20
echo "(vazio acima = garantia de não-persistência OK)"
echo
echo "=== AUDIT: ZIPs em /kaggle/working ==="
find /kaggle/working -type f \( -iname "*.zip" -o -iname "*.7z" -o -iname "*.tar" -o -iname "*.gz" \) 2>/dev/null | head -10
echo "(apenas output_download.zip é esperado durante janela de download ativo)"

In [ ]:
# Push de logs ao Drive (SOMENTE logs — sem imagens, sem ZIP)
# O sync de Drive NÃO inclui /dev/shm (guardrail em kaggle_drive_sync.py)
from kaggle_drive_sync import sync_outputs

log_file = COMFYUI_DIR / "comfyui.log"
has_logs = log_file.exists() and log_file.stat().st_size > 0

if has_logs and DRIVE_AVAILABLE:
    try:
        sync_res = sync_outputs(
            action="push",
            categories=["logs"],
            local_logs=COMFYUI_DIR,  # /kaggle/working/ComfyUI — apenas *.log, *.txt
            drive_base=DRIVE_BASE,
            env="kaggle"
        )
        print(f"[INFO] Push de logs: {sync_res['synced']} enviado(s), {sync_res['skipped']} inalterado(s)")
    except Exception as e:
        print(f"[WARN] Push de logs não pôde ser executado: {e}")
else:
    print("[INFO] Sem log para envio ou Drive indisponível. Pronto para gerar!")
print("[SECURITY] Imagens NÃO residem em /kaggle/working — zero-persistent-image ativo.")

In [ ]:
# =============================================================
# VERIFICAÇÃO FINAL DE FILESYSTEM — execute ao encerrar a sessão
# Falha explicitamente se qualquer imagem ou arquivo sensível
# for encontrado em /kaggle/working
# =============================================================
from comfyui_setup import final_filesystem_check, SecurityError

result = final_filesystem_check(scan_root=WORKDIR, silent=False)

if result["violations"] > 0:
    raise SecurityError(
        f"SECURITY VIOLATION: {result['violations']} artefato(s) sensível(is) "
        "encontrado(s) em /kaggle/working.\n"
        "Esta sessão não é segura. Apague os arquivos listados acima antes de encerrar."
    )

## Próximo passo

O ComfyUI está rodando com **isolamento total de I/O em tmpfs** — nenhum artefato de geração toca `/kaggle/working`.

Os modelos do Dataset são lidos diretamente do Input montado em `/kaggle/input/<slug>` (somente leitura, sem download).

Fluxo operacional por sessão de geração:
1. **Gere** normalmente pela interface ou API (outputs → `/dev/shm/comfy_ui_output`).
2. **Execute a célula de ciclo de vida** (espera fila → ZIP AES-256 em `/dev/shm` → limpa RAM).
3. **Execute a célula de download** (copia ZIP para `/kaggle/working` temporariamente).
4. **Baixe** `/kaggle/working/output_download.zip` pelo painel do Kaggle.
5. **Execute a célula de cleanup pós-download** (apaga staging + ZIP tmpfs + verifica filesystem).
6. **Execute a verificação final** para confirmar zero imagens em `/kaggle/working`.

Para sincronizar **logs** com o Google Drive, use `09_sync_outputs.ipynb`.

## [Opcional] Civitai → Local: adicionar modelo faltante rapidamente

Célula **independente do fluxo principal** (não roda no Run All por padrão — execute manualmente).
Use quando faltar algum modelo no ComfyUI que já está de pé: baixa direto da Civitai por AIR/URL
e salva direto em `MODELS_DIR/<categoria>/` — sem staging, sem manifest e sem publicar no Kaggle Dataset.

Pré-requisitos:
- Já ter rodado a Célula 4 (sync do repositório) nesta sessão.
- Secret `CIVITAI_TOKEN` (ou `CIVITAI_API_KEY`) configurado.

**NOTA DE SEGURANÇA:** Esta célula baixa para `MODELS_DIR` (disco, `/kaggle/working/ComfyUI/models/`)
que é apropriado para binários de modelo. Não baixa imagens.

In [ ]:
# CIVITAI -> LOCAL: adição rápida de modelo (fora do fluxo principal)
os.environ.setdefault("KAGGLE_USERNAME", _get_secret("KAGGLE_USERNAME") or "")
os.environ.setdefault("KAGGLE_DATASET_NAME", _get_secret("KAGGLE_DATASET_NAME") or "")

from kaggle_dataset_manager import (
    CATEGORIES,
    collect_input_queue,
    download_resolved_queue,
    queue_contains_checkpoint,
    resolve_queue_metadata,
)

for _cat in CATEGORIES:
    (MODELS_DIR / _cat).mkdir(parents=True, exist_ok=True)

CIVITAI_TOKEN = _get_secret("CIVITAI_TOKEN") or _get_secret("CIVITAI_API_KEY")
if not CIVITAI_TOKEN:
    raise RuntimeError(
        "CIVITAI_TOKEN não encontrado nos Secrets do Kaggle. "
        "Adicione o Secret CIVITAI_TOKEN antes de rodar esta célula."
    )

print("=" * 60)
print("CIVITAI -> LOCAL (adição rápida de modelo)")
print(f"Destino: {MODELS_DIR}")
print("=" * 60)

_pending = collect_input_queue()

if not _pending:
    print("[INFO] Nenhum item informado; nada a baixar.")
else:
    _resolved = resolve_queue_metadata(_pending, CIVITAI_TOKEN)
    if not _resolved:
        print("[ERROR] Nenhum item válido resolvido; nada a baixar.")
    else:
        _checkpoint_destination = None
        if queue_contains_checkpoint(_resolved):
            _choice = input("Checkpoint: 1=checkpoints/ 2=diffusion_models/: ").strip().lower()
            _checkpoint_destination = {"1": "checkpoints", "2": "diffusion_models"}.get(_choice, _choice)
            print(f"[INFO] Destino de checkpoints deste lote: {_checkpoint_destination}")

        _downloaded = download_resolved_queue(
            _resolved,
            staging_dir=MODELS_DIR,
            token=CIVITAI_TOKEN,
            checkpoint_destination=_checkpoint_destination,
        )

        print(f"\n[SUCCESS] {len(_downloaded)} arquivo(s) adicionados em {MODELS_DIR}")
        for _item in _downloaded:
            print(f"  + {_item.path} ({_item.size / (1024**3):.2f} GB)")